# 5.1 Why Cloudflare Reacts to Scale

---


### **Concept**

When a website uses Cloudflare, incoming requests do not go directly to the website’s server.  
Instead, they first pass through Cloudflare, which analyzes traffic before allowing it through.

Cloudflare monitors traffic patterns to detect suspicious activity such as automated scraping or denial-of-service attacks.

When scraping operations scale up, request patterns can begin to look automated rather than human. 

---

### **What “Scale” Looks Like to Cloudflare**

From Cloudflare’s perspective, scaling often appears as:
- large numbers of requests from the same IP address
- very fast request frequency
- repeated requests to similar endpoints (pagination)
- identical headers across many requests
- predictable request timing

Example pattern:


    /products?page=1
    /products?page=2
    /products?page=3
    /products?page=4
    /products?page=5

If these requests occur very quickly, the pattern becomes easy to detect.

---

### **Common Reactions from Cloudflare**

When traffic appears suspicious, Cloudflare may respond with:<br><br>

**429 – Too Many Requests**  

    HTTP 429

This indicates rate limiting.  <br><br>

**403 – Forbidden**

    HTTP 403

This means access has been denied.  <br><br>

**Error 1020**

    Cloudflare-specific block triggered by security rules.  

<br>

**JavaScript Challenge**

    Checking your browser before accessing...

The browser must execute JavaScript to continue.

---

<br>

<br>


# 5.2 Rate Limiting Strategies

---

### **Concept**
Many websites restrict how frequently a client can send requests.

This is called rate limiting.

Rate limits protect servers from excessive traffic and ensure fair usage of resources.

If a scraper sends requests too quickly, the server may temporarily block or slow down access.

---

### **Common Rate Limit Responses**

Servers often indicate rate limits using specific HTTP status codes.

429 – Too Many Requests

    HTTP 429

This means the client has exceeded the allowed request rate.

Some APIs may also include headers such as:

    Retry-After: 60

This indicates how long the client should wait before sending another request.

---

### **Strategy 1 — Add Request Delays**

Instead of sending requests as fast as possible, introduce delays between them.

Example approach:

    1 request every 2 seconds

This reduces the likelihood of triggering rate limits.

<br>

### **Practical Lab — Controlled Request Rate**

Try sending multiple requests to a public API while controlling request timing.



In [1]:
import requests

import time

url = "https://jsonplaceholder.typicode.com/posts"

for i in range(5):
    response = requests.get(url)
    print("Request", i+1, "Status:", response.status_code)
    time.sleep(2)

Request 1 Status: 200
Request 2 Status: 200
Request 3 Status: 200
Request 4 Status: 200
Request 5 Status: 200


This script sends one request every two seconds.



---

### **Strategy 2 — Detect Rate Limit Responses**

Scrapers should monitor responses and react appropriately.

Example handling of a 429 response:



In [2]:
import requests
import time

url = "https://httpbin.org/status/429"

response = requests.get(url)

if response.status_code == 429:
    print("Rate limit detected. Waiting before retrying...")
    time.sleep(10)

Rate limit detected. Waiting before retrying...


Instead of continuing aggressively, the scraper pauses before retrying.

---

### **Strategy 3 — Reduce Concurrent Requests**

Making many requests at the same time can trigger rate limits quickly.

Reducing concurrency helps maintain stable access.

For example:

    Instead of 10 requests at once
    send 1–2 requests at a time

---

<br>

<br>


# 5.3 Residential vs Datacenter Proxies

---

### **Concept**

Websites track the IP address of incoming requests.
If many requests originate from the same IP, especially from cloud infrastructure, the traffic may appear automated.

Proxies allow requests to be routed through another server, so the destination website sees the proxy’s IP instead of the client’s.

This helps distribute traffic across different IP addresses.

Two common proxy types are datacenter proxies and residential proxies.

---

### **Datacenter Proxies**

Datacenter proxies come from cloud infrastructure providers.


**Characteristics:**

- very fast
- inexpensive
- easy to obtain
- easier for anti-bot systems to detect

These IPs often belong to known hosting providers.

---

### **Residential Proxies**

Residential proxies use IP addresses assigned to home internet connections.

**Characteristics:**

- appear as normal users
- harder to detect
- slower
- more expensive

Because they resemble normal user traffic, they often encounter fewer restrictions.

---

### **Practical Lab — Check Your Current IP**

Run this first to see your real IP.

In [3]:
import requests

response = requests.get("https://httpbin.org/ip")
print("Your IP:", response.json())

Your IP: {'origin': '106.219.159.235'}


---

### **Practical Lab — Using a Real Public Proxy**

Below are few public proxies that can be tested.

In [10]:
import requests

def get_working_proxy():
    """
    Fetch a free proxy and test it
    """
    url = "https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/http.txt"
    
    try:
        proxy_list = requests.get(url, timeout=10).text.splitlines()
    except:
        return None

    for proxy in proxy_list[:20]:  # try first 20 proxies
        proxies = {
            "http": f"http://{proxy}",
            "https": f"http://{proxy}",
        }

        try:
            response = requests.get(
                "https://httpbin.org/ip",
                proxies=proxies,
                timeout=3,
                headers={
                    "User-Agent": "Mozilla/5.0"
                }
            )

            if response.status_code == 200:
                print(f"✅ Working proxy found: {proxy}")
                return proxies

        except:
            continue

    return None


def main():
    print("🔹 Checking without proxy...")
    normal_ip = requests.get("https://httpbin.org/ip").json()
    print("Your IP:", normal_ip)

    print("\n🔹 Finding working proxy...")
    proxy = get_working_proxy()

    if not proxy:
        print("❌ No working proxy found. Try again.")
        return

    print("\n🔹 Checking with proxy...")
    proxied_ip = requests.get(
        "https://httpbin.org/ip",
        proxies=proxy,
        timeout=5
    ).json()

    print("Proxy IP:", proxied_ip)


if __name__ == "__main__":
    main()

🔹 Checking without proxy...
Your IP: {'origin': '106.219.159.235'}

🔹 Finding working proxy...
❌ No working proxy found. Try again.


In [ ]:
import requests
import random

# get proxy list
url = "https://proxylist.geonode.com/api/proxy-list?limit=10&page=1"

data = requests.get(url).json()

proxy = random.choice(data["data"])

proxy_address = proxy["ip"] + ":" + proxy["port"]

proxies = {
    "http": "http://" + proxy_address,
    "https": "http://" + proxy_address
}

print("Using proxy:", proxy_address)

response = requests.get("https://httpbin.org/ip", proxies=proxies, timeout=10)

print("Server sees IP:", response.json())

In [19]:
import requests
import random

# get proxy list
url = "https://proxylist.geonode.com/api/proxy-list?limit=20&page=1"

data = requests.get(url).json()["data"]

print("Fetched", len(data), "proxies\n")

for proxy in data:

    proxy_address = f"{proxy['ip']}:{proxy['port']}"

    proxies = {
        "http": "http://" + proxy_address,
        "https": "http://" + proxy_address
    }

    print("Testing proxy:", proxy_address)

    try:

        r = requests.get(
            "https://httpbin.org/ip",
            proxies=proxies,
            timeout=5
        )

        print("Working proxy!")
        print("Server sees:", r.json())

        break

    except Exception:
        print("Failed\n")

Fetched 20 proxies

Testing proxy: 62.109.31.192:20000
Failed

Testing proxy: 173.249.31.157:28187
Failed

Testing proxy: 212.200.89.10:4153
Failed

Testing proxy: 46.227.37.149:1088
Failed

Testing proxy: 110.78.146.11:4145
Failed

Testing proxy: 147.135.36.162:57752
Failed

Testing proxy: 5.135.191.56:56750
Failed

Testing proxy: 5.78.70.186:8080
Failed

Testing proxy: 200.152.107.98:1080
Failed

Testing proxy: 111.193.226.62:23456
Failed

Testing proxy: 41.85.189.66:39475
Failed

Testing proxy: 162.255.108.249:5678
Failed

Testing proxy: 117.10.124.11:1080
Failed

Testing proxy: 183.177.127.42:5678
Failed

Testing proxy: 157.185.173.217:26589
Failed

Testing proxy: 89.204.80.190:4153
Failed

Testing proxy: 103.134.180.97:4153
Failed

Testing proxy: 68.103.98.161:36081
Failed

Testing proxy: 138.97.0.173:4145
Failed

Testing proxy: 118.67.170.121:4153
Failed



In [14]:
import requests

def fetch_ip(session):
    return session.get("https://httpbin.org/ip", timeout=5).json()


# Normal session (no proxy)
normal_session = requests.Session()

print("🔹 Without proxy:")
print(fetch_ip(normal_session))


# "Proxy-configured" session (demonstration)
proxy_session = requests.Session()

proxy_session.proxies = {
    "http": "http://dummy-proxy:8080",
    "https": "http://dummy-proxy:8080"
}

print("\n🔹 With proxy configured:")
print("Proxies set to:", proxy_session.proxies)

# Still works because we are demonstrating configuration, not relying on dead proxies
print(fetch_ip(normal_session))

🔹 Without proxy:
{'origin': '106.219.159.235'}

🔹 With proxy configured:
Proxies set to: {'http': 'http://dummy-proxy:8080', 'https': 'http://dummy-proxy:8080'}
{'origin': '106.219.159.235'}


In [ ]:
import requests

def get_ip(url):
    return requests.get(url, timeout=5).json()

# Normal request
print("🔹 Without proxy:")
print(get_ip("https://httpbin.org/ip"))

# Request via proxy service
print("\n🔹 With proxy (via AllOrigins):")

proxy_url = "https://api.allorigins.win/raw?url=https://httpbin.org/ip"

print(get_ip(proxy_url))

Above were a few examples of using public proxies. However, public proxies are often highly unreliable, which is why using Tor or Cloudflare is preferred.

---